# Layer 4 — Pelatihan pada Januari–Februari dan Uji pada Maret

Pembagian sudah terjadi di Layer 3 menurut waktu, bukan menurut acak.

1. SMOTE hanya menyeimbangkan data latih Januari–Februari.
2. Data uji Maret tetap berisi struk sungguhan.
3. Keempat model dilatih pada seluruh data latih setelah SMOTE, tanpa cuplikan. SVM tidak ikut karena probabilitasnya tidak praktis pada puluhan ribu baris.
4. XGBoost yang disimpan adalah model integrasi data riil: `models/xgboost_cross_sell_ril.pkl`. Berkas model Instacart tidak ditimpa.


In [1]:
from pathlib import Path
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")

def find_project_dir() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "outputs_ril" / "layer3_train_features.csv").exists():
            return candidate
    raise FileNotFoundError("Jalankan Layer 3 terlebih dahulu.")

PROJECT_DIR = find_project_dir()
TRAIN_PATH = PROJECT_DIR / "outputs_ril" / "layer3_train_features.csv"
TEST_PATH = PROJECT_DIR / "outputs_ril" / "layer3_test_features.csv"
MODEL_PATH = PROJECT_DIR / "models" / "xgboost_cross_sell_ril.pkl"
COMPARISON_PATH = PROJECT_DIR / "outputs_ril" / "layer4_model_comparison.csv"
CM_PATH = PROJECT_DIR / "outputs_ril" / "layer4_xgboost_confusion_matrix.png"
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

FEATURE_COLUMNS = [
    "current_basket_size", "hour_of_day", "day_of_week", "is_weekend",
    "rule_confidence", "rule_lift", "antecedent_rate", "interest_lift",
]
RANDOM_STATE = 42

print("--> [INFO] Parameter Layer 4 siap. SMOTE hanya pada data latih. Uji = Maret.")
print(f"--> [INFO] Folder proyek: {PROJECT_DIR}")


--> [INFO] Parameter Layer 4 siap. SMOTE hanya pada data latih. Uji = Maret.
--> [INFO] Folder proyek: /Users/stefanieagahari/Downloads/Cross Selling Retail


## 1. SMOTE pada latih, uji tetap murni


In [2]:
print("--> [INFO] Memuat fitur latih dan uji, lalu menyeimbangkan hanya data latih...")
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
X_train = train_df[FEATURE_COLUMNS].astype("float32")
y_train = train_df["y"].astype("int8")
X_test = test_df[FEATURE_COLUMNS].astype("float32")
y_test = test_df["y"].astype("int8")
print("latih sebelum SMOTE", X_train.shape, "proporsi 1", round(float(y_train.mean()), 4))
print("uji", X_test.shape, "proporsi 1", round(float(y_test.mean()), 4))

sampler = SMOTE(random_state=RANDOM_STATE, k_neighbors=min(5, int(y_train.sum()) - 1))
X_bal, y_bal = sampler.fit_resample(X_train, y_train)
print("latih sesudah SMOTE", X_bal.shape, "proporsi 1", round(float(np.mean(y_bal)), 4))

X_fit, y_fit = X_bal, y_bal
print(f"--> [INFO] Seluruh latih dipakai: {len(X_fit):,} baris")


--> [INFO] Memuat fitur latih dan uji, lalu menyeimbangkan hanya data latih...
latih sebelum SMOTE (42946, 8) proporsi 1 0.2359
uji (22046, 8) proporsi 1 0.243
latih sesudah SMOTE (65626, 8) proporsi 1 0.5
--> [INFO] Seluruh latih dipakai: 65,626 baris


## 2. Empat model pada seluruh data latih

Metrik dihitung pada seluruh struk Maret. SVM tidak dilatih.


In [3]:
print("--> [INFO] Melatih empat model dan mengevaluasi data uji Maret...")
models = {
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, objective="binary:logistic",
        eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist",
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=200, learning_rate=0.1, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE,
    ),
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]),
}

rows = []
fitted = {}
for name, model in models.items():
    start = time.perf_counter()
    model.fit(X_fit, y_fit)
    fitted[name] = model
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    rows.append({
        "Model": name,
        "Akurasi": accuracy_score(y_test, pred),
        "Precision Macro": precision_score(y_test, pred, average="macro", zero_division=0),
        "Recall Macro": recall_score(y_test, pred, average="macro", zero_division=0),
        "F1 Macro": f1_score(y_test, pred, average="macro", zero_division=0),
        "AUC-ROC": roc_auc_score(y_test, proba),
        "Detik": round(time.perf_counter() - start, 1),
    })
    print(name, "selesai")

comparison = pd.DataFrame(rows).sort_values("F1 Macro", ascending=False)
comparison.to_csv(COMPARISON_PATH, index=False)
print(comparison.to_string(index=False))

xgb = fitted["XGBoost"]
pred_xgb = xgb.predict(X_test)
print(classification_report(y_test, pred_xgb, digits=4, zero_division=0))
matrix = confusion_matrix(y_test, pred_xgb)
fig, ax = plt.subplots(figsize=(4.5, 4))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_xlabel("Prediksi")
ax.set_ylabel("Aktual")
ax.set_title("XGBoost pada struk Maret")
fig.tight_layout()
fig.savefig(CM_PATH, dpi=140)
plt.close(fig)
joblib.dump(xgb, MODEL_PATH)
print("--> [INFO] Model tersimpan:", MODEL_PATH)


--> [INFO] Melatih empat model dan mengevaluasi data uji Maret...
XGBoost selesai
LightGBM selesai
Random Forest selesai
Logistic Regression selesai
              Model  Akurasi  Precision Macro  Recall Macro  F1 Macro  AUC-ROC  Detik
      Random Forest 0.713236         0.638759      0.664165  0.645618 0.741467    1.4
            XGBoost 0.769074         0.675734      0.630658  0.643538 0.739624    0.6
Logistic Regression 0.680758         0.627927      0.664444  0.629094 0.731340    0.1
           LightGBM 0.773927         0.685346      0.612386  0.625892 0.739925    1.5
              precision    recall  f1-score   support

           0     0.8144    0.9000    0.8551     16688
           1     0.5370    0.3613    0.4320      5358

    accuracy                         0.7691     22046
   macro avg     0.6757    0.6307    0.6435     22046
weighted avg     0.7470    0.7691    0.7523     22046

--> [INFO] Model tersimpan: /Users/stefanieagahari/Downloads/Cross Selling Retail/models/xgboo